# Project Overview: Environmental Comfort Prediction Pipeline

Welcome to the data engineering pipeline for the Environmental Comfort Prediction project. Our goal is to consolidate various isolated IoT sensor datasets into a single, machine-learning-ready database to predict human comfort and focus scores.

We initially sourced **8 distinct environmental datasets** from Kaggle and Zenodo. To transform these raw files into a functional predictive model, this project is structured into a sequential pipeline:

* **Notebook 1: Data Preparation** (Cleaning structurally broken raw datasets).
* **Notebook 2: Pre-Merge EDA** (Analyzing the 8 datasets, and selecting the highest-quality sources for unification).
* **Pipeline Script:** `build_unified_environment_dataset.py` (The automated engine that aligns timestamps and merges our chosen datasets into a 1M+ row CSV).
* **Notebook 3: Unified Dataset Analysis** (Validating the integrity of the merged data and identifying missing targets).
* **Notebook 4 & 5: ML Imputation & Linearization** (Using predictive models to fill missing ground-truth labels and formatting the time-series blocks for final classification).

# Phase 1: Data Preparation & Cleaning

Before we can analyze our datasets and merge them into a single unified timeline, we must address critical structural issues in two of our raw data sources: **Data 7 (HomeCoach)** and **Data 6 (KETI)**.

This notebook documents the data engineering steps required to clean these datasets and move them from the `raw/` folder to the `interim/` folder.

### 1. The HomeCoach Dataset (Data 7)
**The Problem:** The raw HomeCoach data is split across four separate CSV files, divided by year (2023, 2024, 2025, 2026).
**The Solution:** We must concatenate these files vertically, parse their timestamps, sort them chronologically, and assign a standard `id` column.

### 2. The KETI Dataset (Data 6)
**The Problem:** The raw KETI dataset is highly disjointed. The environmental sensors (Temperature, Humidity, Light, CO2) sample every 5 seconds, while the PIR (Motion) sensor samples every 10 seconds. Joining them on exact timestamps results in a massive file full of misaligned, staggered rows.
**The Solution:** We must downsample the data into **1-minute windows**. We will take the average (`mean`) of the environmental sensors during that minute, and the maximum (`max`) of the PIR sensor (so if motion happened at all during that minute, it registers as a 1). We also filter out impossible sensor glitches (e.g., temperature reading 500°C).

### Executing the Data Engineering Scripts

Because processing millions of raw sensor rows is memory-intensive, we handle this using dedicated Python scripts rather than Jupyter Notebooks. 

**To prepare the data, open your terminal and run:**
1. `python scripts/prepare_homecoach.py`
2. `python scripts/resample_keti_data.py`

*(Note: The scripts folder assumes you have saved the provided `.py` files into a `scripts/` directory in your project root).*

Below, we load the outputs to verify the scripts executed successfully.

In [ ]:
import pandas as pd
from pathlib import Path

# Verify HomeCoach
homecoach_path = '../../data/interim/HomeCoach_combined.csv'
try:
    df_hc = pd.read_csv(homecoach_path)
    print(f" !!!!OK!!!! HomeCoach Interim Data Loaded Successfully!")
    print(f"Total Rows: {len(df_hc):,}")
    print(f"Date Range: {df_hc['timestamp'].min()} to {df_hc['timestamp'].max()}")
    display(df_hc.head())
except FileNotFoundError:
    print(f"!!!!NOT OK!!!! File not found. Please run prepare_homecoach.py in the terminal.")

 !!!!OK!!!! HomeCoach Interim Data Loaded Successfully!
Total Rows: 311,692
Date Range: 2023-03-04 09:12:50 to 2026-05-06 15:47:49


,id,timestamp,Temperature,Humidity,CO2,Noise,Pressure
0,HomeCoach,2023-03-04 09:12:50,17.7,56,2305,67,1006.6
1,HomeCoach,2023-03-04 09:17:50,17.9,56,2250,73,1006.6
2,HomeCoach,2023-03-04 09:22:50,18.3,57,2286,77,1006.4
3,HomeCoach,2023-03-04 09:27:50,18.9,57,2266,58,1006.4
4,HomeCoach,2023-03-04 09:32:50,19.1,57,2238,54,1006.3


In [12]:
# Verify KETI
keti_path = '../../data/interim/keti_1min_resampled.csv'
try:
    df_keti = pd.read_csv(keti_path)
    print(f"!!!!OK!!!! KETI Resampled Interim Data Loaded Successfully!")
    print(f"Total Rows: {len(df_keti):,}")
    print(f"Unique Rooms: {df_keti['room'].nunique()}")
    display(df_keti.head())
except FileNotFoundError:
    print(f"!!!!NOT OK!!!! File not found. Please run resample_keti_data.py in the terminal.")

!!!!OK!!!! KETI Resampled Interim Data Loaded Successfully!
Total Rows: 604,891
Unique Rooms: 53


C:\Users\alsal\AppData\Local\Temp\ipykernel_15100\1655340606.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_keti = pd.read_csv(keti_path)


,room,timestamp,co2,humidity,light,temperature,pir,co2_observations,humidity_observations,light_observations,temperature_observations,pir_observations
0,413,2013-08-23 23:05:00,494.727273,45.3300,96.555556,23.927778,0.0,11,9,9,9,7
1,413,2013-08-23 23:06:00,496.666667,45.3275,96.833333,23.938333,0.0,12,12,12,12,6
2,413,2013-08-23 23:07:00,498.916667,45.3250,97.333333,23.942500,0.0,12,12,12,12,6
3,413,2013-08-23 23:08:00,504.166667,45.3125,97.083333,23.952500,0.0,12,12,12,12,6
4,413,2013-08-23 23:09:00,499.500000,45.3100,97.416667,23.955833,0.0,12,12,12,12,6


---
### Next Steps
With our raw data cleaned and standardized into the `interim/` folder, we can now proceed to **`2_pre_merge_eda.ipynb`** to analyze the distributions of our datasets before deciding which of the sets we would like to combine into our final unified dataset.